# TH01 – Python/Pandas for Business Analytics
**Môn:** Phân tích dữ liệu lớn trong đầu tư và kinh doanh

## Mục tiêu
Sau bài này, sinh viên có thể:
- sử dụng Python cơ bản cho phân tích dữ liệu;
- thao tác với NumPy và Pandas;
- đọc, kiểm tra và làm sạch dữ liệu kinh doanh;
- tính các KPI bán hàng;
- phân tích theo thời gian, sản phẩm, khu vực, kênh và khách hàng;
- trực quan hóa và viết business insights có số liệu kiểm chứng.


## 0. Chuẩn bị môi trường


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

DATA = Path("data")
FILE = DATA / "business_sales_dirty.csv"


## 1. Python cơ bản cho Data Analyst


In [ ]:
# List, dictionary, comprehension, function
sales = [120, 150, 90, 200, 175]
print("Total:", sum(sales))
print("Average:", sum(sales)/len(sales))

high_sales = [x for x in sales if x >= 150]
print("High sales:", high_sales)

def growth_rate(current, previous):
    if previous == 0:
        return np.nan
    return (current - previous) / previous

print("Growth:", growth_rate(120, 100))


### Bài 1
1. Tạo list doanh thu 12 tháng.
2. Tính tổng, trung bình, min, max.
3. Viết hàm tính tăng trưởng giữa hai tháng.
4. Dùng list comprehension để lấy các tháng có doanh thu cao hơn trung bình.


## 2. NumPy cho phân tích kinh doanh


In [ ]:
revenue = np.array([100, 120, 90, 150, 160, 140], dtype=float)

print("Mean:", revenue.mean())
print("Std :", revenue.std())
print("Min :", revenue.min())
print("Max :", revenue.max())

returns = revenue[1:] / revenue[:-1] - 1
print("Growth rates:", returns)


### Bài 2
Cho mảng doanh thu theo tháng:
- tính mean, median, standard deviation;
- tính tăng trưởng tháng;
- xác định tháng tăng trưởng cao nhất/thấp nhất.


## 3. Đọc dữ liệu bằng Pandas


In [ ]:
df = pd.read_csv(FILE)

print(df.shape)
display(df.head())
df.info()


### Bài 3 – Data Understanding


In [ ]:
display(df.describe(include="all").T)
print("Unique customers:", df["CustomerID"].nunique())
print("Unique products :", df["ProductID"].nunique())
print("Date range:", df["TransactionDate"].min(), "→", df["TransactionDate"].max())


## 4. Data Quality Audit


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "unique": df.nunique(dropna=True)
})
display(quality)

print("Duplicate rows:", df.duplicated().sum())

invalid = df[
    (df["UnitPrice"] <= 0) |
    (df["Discount"] < 0) |
    (df["Discount"] > 1) |
    (df["Quantity"] == 0)
]
display(invalid.head(20))


### Bài 4 – Phân biệt lỗi và nghiệp vụ
Giải thích:
1. UnitPrice âm có hợp lệ không?
2. Discount > 1 có hợp lệ không?
3. Quantity âm có luôn là lỗi không?
4. Missing CustomerID nên xóa, điền hay giữ?
5. Category viết hoa/thường khác nhau ảnh hưởng gì đến phân tích?


## 5. Data Cleaning


In [ ]:
clean = df.copy()

clean["TransactionDate"] = pd.to_datetime(clean["TransactionDate"], errors="coerce")

clean = clean.drop_duplicates(subset=["TransactionID"])
clean = clean[clean["UnitPrice"].notna() & (clean["UnitPrice"] > 0)]
clean = clean[clean["Discount"].between(0, 1)]
clean = clean[clean["Quantity"] != 0]

clean["Category"] = clean["Category"].str.strip().str.title()
clean["Region"] = clean["Region"].fillna("Unknown")
clean["Rating"] = pd.to_numeric(clean["Rating"], errors="coerce")

print("Before:", len(df))
print("After :", len(clean))


## 6. Tính KPI kinh doanh


In [ ]:
sales = clean[clean["Revenue"] > 0].copy()

kpis = {
    "Revenue": sales["Revenue"].sum(),
    "Profit": sales["Profit"].sum(),
    "Transactions": sales["TransactionID"].nunique(),
    "Customers": sales["CustomerID"].nunique(),
    "Units": sales["Quantity"].sum(),
}
kpis["AOV"] = kpis["Revenue"] / kpis["Transactions"]
kpis["Profit Margin"] = kpis["Profit"] / kpis["Revenue"]

pd.Series(kpis).round(2)


### Bài 5
Tính và giải thích:
- Revenue
- Profit
- Number of Transactions
- Number of Customers
- Units Sold
- Average Order Value
- Profit Margin
- Average Revenue per Customer


## 7. Phân tích theo thời gian


In [ ]:
sales["YearMonth"] = sales["TransactionDate"].dt.to_period("M").astype(str)
sales["Weekday"] = sales["TransactionDate"].dt.day_name()

monthly = sales.groupby("YearMonth", as_index=False).agg(
    Revenue=("Revenue","sum"),
    Profit=("Profit","sum"),
    Transactions=("TransactionID","nunique"),
    Customers=("CustomerID","nunique")
)
monthly["MoM_Growth"] = monthly["Revenue"].pct_change()
display(monthly.head())

plt.figure(figsize=(10,4))
plt.plot(monthly["YearMonth"], monthly["Revenue"], marker="o")
plt.xticks(rotation=45)
plt.title("Monthly Revenue")
plt.tight_layout()
plt.show()


## 8. Product / Category / Region / Channel Analytics


In [ ]:
category = sales.groupby("Category", as_index=False).agg(
    Revenue=("Revenue","sum"),
    Profit=("Profit","sum"),
    Units=("Quantity","sum")
)
category["Margin"] = category["Profit"] / category["Revenue"]
display(category.sort_values("Revenue", ascending=False))

region_channel = pd.pivot_table(
    sales,
    index="Region",
    columns="Channel",
    values="Revenue",
    aggfunc="sum",
    fill_value=0
)
display(region_channel)


### Bài 6 – Business questions
1. Category nào có doanh thu cao nhất?
2. Category nào có margin cao nhất?
3. Region nào đóng góp doanh thu cao nhất?
4. Channel nào hoạt động tốt nhất?
5. Có region-channel combination nào yếu rõ rệt không?


## 9. Customer Analytics cơ bản


In [ ]:
customer = sales.groupby("CustomerID", as_index=False).agg(
    Revenue=("Revenue","sum"),
    Profit=("Profit","sum"),
    Transactions=("TransactionID","nunique"),
    AvgRating=("Rating","mean"),
    LastPurchase=("TransactionDate","max")
)

customer["AOV"] = customer["Revenue"] / customer["Transactions"]
display(customer.sort_values("Revenue", ascending=False).head(10))


### Pareto 80/20


In [ ]:
cust_sorted = customer.sort_values("Revenue", ascending=False).copy()
cust_sorted["CumRevenue"] = cust_sorted["Revenue"].cumsum()
cust_sorted["CumShare"] = cust_sorted["CumRevenue"] / cust_sorted["Revenue"].sum()

n80 = (cust_sorted["CumShare"] < 0.8).sum() + 1
share_customers = n80 / len(cust_sorted)

print("Customers needed for ~80% revenue:", n80)
print("Share of customers:", round(share_customers*100,2), "%")


## 10. Merge dữ liệu tổng hợp


In [ ]:
customer_region = (
    sales.groupby(["CustomerID","Region"], as_index=False)["Revenue"].sum()
    .sort_values(["CustomerID","Revenue"], ascending=[True,False])
    .drop_duplicates("CustomerID")
)

customer360 = customer.merge(
    customer_region[["CustomerID","Region"]],
    on="CustomerID",
    how="left"
)

display(customer360.head())


## 11. Visualization cho Business Analytics


In [ ]:
top_cat = category.sort_values("Revenue", ascending=False)

plt.figure(figsize=(8,4))
plt.bar(top_cat["Category"], top_cat["Revenue"])
plt.xticks(rotation=45)
plt.title("Revenue by Category")
plt.tight_layout()
plt.show()

region_rev = sales.groupby("Region")["Revenue"].sum().sort_values(ascending=False)
plt.figure(figsize=(6,4))
plt.bar(region_rev.index, region_rev.values)
plt.title("Revenue by Region")
plt.tight_layout()
plt.show()


### Bài 7 – Thiết kế biểu đồ
Tạo tối thiểu 4 biểu đồ:
1. Revenue theo tháng.
2. Revenue theo Category.
3. Profit Margin theo Category.
4. Revenue theo Region hoặc Channel.

Mỗi biểu đồ phải có 1–2 câu giải thích ý nghĩa kinh doanh.


## 12. Mini Business Analytics Challenge


In [ ]:
# Gợi ý:
# 1. Top 10 customers theo Revenue
# 2. Top 10 products theo Revenue
# 3. Tháng có Revenue và Profit cao nhất
# 4. Region có AOV cao nhất
# 5. Channel có Profit Margin cao nhất
# 6. Tỷ lệ doanh thu từ top 10% khách hàng


## 13. Executive Summary
Sinh viên viết tối đa 200 từ, bắt buộc gồm:
- 5 phát hiện có số liệu;
- 3 đề xuất kinh doanh;
- 2 hạn chế của dataset hoặc phân tích.

**Không đưa ra insight nếu không có số liệu kiểm chứng từ notebook.**


## 14. Yêu cầu nộp
- `TH01_HoTen_MSSV.ipynb`
- kết quả chạy đầy đủ;
- tối thiểu 4 biểu đồ;
- 5 business insights;
- 3 recommendations;
- trả lời toàn bộ câu hỏi thảo luận.
